# 12-target-language LAPE identity comparison

LAPE selection uses 13 languages including English; reporting uses the same 12 non-English targets as transfer. Reuses the existing FP16 13-language rates and three quantized 6-language rates; computes only the missing seven languages per quantizer. Run on a GPU server.

In [1]:
import gc, json, math, os, random, subprocess, sys, time, warnings
from collections import defaultdict
from pathlib import Path

# AutoDL/中国大陆网络：必须在导入 transformers/huggingface_hub 之前设置。
os.environ.setdefault('HF_ENDPOINT', 'https://hf-mirror.com')
os.environ.setdefault('HF_HOME', '/root/autodl-tmp/huggingface')
os.environ.setdefault('HF_DATASETS_CACHE', '/root/autodl-tmp/huggingface/datasets')
os.environ.setdefault('HF_HUB_DISABLE_XET', '1')
os.environ.setdefault('HF_HUB_ETAG_TIMEOUT', '30')
os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT', '120')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

torch.set_grad_enabled(False)


libgomp: Invalid value for environment variable OMP_NUM_THREADS


In [2]:
# Reuse the exact Wikipedia discovery blocks used by the saved 6-language run.
FP16_CHECKPOINT = Path('/root/autodl-tmp/models/llama3.1-8b/llama3.1-8b-instruct')
QUANT_SPECS = {
    'AWQ-INT4': {'kind': 'checkpoint', 'checkpoint': Path('/root/autodl-tmp/models/llama3.1-8b/llama3.1-8b-awq-int4')},
    'GPTQ-INT4': {'kind': 'checkpoint', 'checkpoint': Path('/root/autodl-tmp/models/llama3.1-8b/llama3.1-8b-gptq-int4')},
    'BNB-NF4': {'kind': 'bnb_nf4', 'checkpoint': FP16_CHECKPOINT},
}
ACTIVE_QUANT_METHODS = ['AWQ-INT4', 'GPTQ-INT4', 'BNB-NF4']
ACTIVE_QUANT_SPECS = QUANT_SPECS
LANGUAGES = ['ca', 'en', 'es', 'fr', 'hu', 'nl', 'tr', 'ar', 'el', 'he', 'ja', 'ko', 'zh']
TARGET_LANGUAGES = [language for language in LANGUAGES if language != 'en']
OLD_LANGUAGES = ['en', 'es', 'tr', 'ar', 'he', 'zh']
OLD_DIR = Path('/root/autodl-tmp/analysis/paper_wikipedia_lape_quantization')
OUTPUT_DIR = Path('/root/autodl-tmp/analysis/paper_wikipedia_lape_13languages_quantized_full_layers')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WIKI_DISCOVERY_TOKENS_PER_LANGUAGE = 1_000_000
WIKI_HELDOUT_TOKENS_PER_LANGUAGE = 100_000
WIKI_SEQUENCE_LENGTH = 512
WIKI_STATS_BATCH_SIZE = 4
LAPE_BOTTOM_RATE = 0.10
ACTIVATION_PROBABILITY_QUANTILE = 0.95
DTYPE = torch.float16
DEVICE_MAP = {'': 0}
TRUST_REMOTE_CODE = True
REQUIRE_FULL_GPU_RESIDENCY = True
MAX_LENGTH = 512
print('LAPE identification languages:', LANGUAGES)
print('Plotted target languages:', TARGET_LANGUAGES)


LAPE identification languages: ['ca', 'en', 'es', 'fr', 'hu', 'nl', 'tr', 'ar', 'el', 'he', 'ja', 'ko', 'zh']
Plotted target languages: ['ca', 'es', 'fr', 'hu', 'nl', 'tr', 'ar', 'el', 'he', 'ja', 'ko', 'zh']


In [3]:
def decoder_layers(model):
    for path in ('model.layers', 'model.decoder.layers', 'transformer.h', 'gpt_neox.layers'):
        obj = model; ok = True
        for part in path.split('.'):
            if not hasattr(obj, part):
                ok = False; break
            obj = getattr(obj, part)
        if ok:
            return obj
    raise AttributeError('Decoder layers not found.')

def input_device(model):
    return model.get_input_embeddings().weight.device

def load_tokenizer():
    tok = AutoTokenizer.from_pretrained(FP16_CHECKPOINT, trust_remote_code=TRUST_REMOTE_CODE)
    tok.padding_side = 'right'
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    return tok

def load_model(name):
    if name == 'FP16':
        checkpoint = FP16_CHECKPOINT; kwargs = {'torch_dtype': DTYPE}
    else:
        spec = ACTIVE_QUANT_SPECS[name]; checkpoint = Path(spec['checkpoint'])
        if spec['kind'] == 'bnb_nf4':
            kwargs = {'quantization_config': BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=DTYPE,
            )}
        else:
            kwargs = {'torch_dtype': DTYPE}
    if not Path(checkpoint).exists():
        raise FileNotFoundError(f'Missing checkpoint: {checkpoint}')
    model = AutoModelForCausalLM.from_pretrained(
        checkpoint, device_map=DEVICE_MAP, low_cpu_mem_usage=True,
        trust_remote_code=TRUST_REMOTE_CODE, **kwargs,
    ).eval()
    model.config.use_cache = False
    return model

def encode_prompt(tokenizer, prompt, device):
    encoded = tokenizer(prompt, return_tensors='pt', truncation=True,
                        max_length=MAX_LENGTH, add_special_tokens=True)
    return {k: v.to(device) for k, v in encoded.items()}

def wiki_cache_path(language):
    total = WIKI_DISCOVERY_TOKENS_PER_LANGUAGE + WIKI_HELDOUT_TOKENS_PER_LANGUAGE
    return OUTPUT_DIR / f'wikipedia_tokens_{language}_{total}_seq{WIKI_SEQUENCE_LENGTH}.pt'

def build_wikipedia_blocks(tokenizer, language):
    cache = wiki_cache_path(language)
    if cache.exists() and not REBUILD_WIKI_TOKEN_CACHE:
        payload = torch.load(cache, map_location='cpu')
        print('Loaded:', cache, tuple(payload['discovery'].shape), tuple(payload['heldout'].shape))
        return payload
    required = WIKI_DISCOVERY_TOKENS_PER_LANGUAGE + WIKI_HELDOUT_TOKENS_PER_LANGUAGE
    target_blocks = math.ceil(required / WIKI_SEQUENCE_LENGTH)
    blocks, buffer = [], []
    for text in tqdm(wikipedia_text_stream(language), desc=f'tokenize-wiki:{language}'):
        ids = tokenizer(text, add_special_tokens=False, truncation=False)['input_ids']
        if ids:
            buffer.extend(ids)
        while len(buffer) >= WIKI_SEQUENCE_LENGTH and len(blocks) < target_blocks:
            blocks.append(buffer[:WIKI_SEQUENCE_LENGTH])
            del buffer[:WIKI_SEQUENCE_LENGTH]
        if len(blocks) >= target_blocks:
            break
    discovery_blocks = WIKI_DISCOVERY_TOKENS_PER_LANGUAGE // WIKI_SEQUENCE_LENGTH
    heldout_blocks = WIKI_HELDOUT_TOKENS_PER_LANGUAGE // WIKI_SEQUENCE_LENGTH
    if len(blocks) < discovery_blocks + heldout_blocks:
        raise RuntimeError(f'Wikipedia {language}: insufficient tokens; got {len(blocks)} blocks.')
    tensor = torch.tensor(blocks[:discovery_blocks + heldout_blocks], dtype=torch.long)
    payload = {
        'discovery': tensor[:discovery_blocks].contiguous(),
        'heldout': tensor[discovery_blocks:discovery_blocks + heldout_blocks].contiguous(),
        'language': language,
        'sequence_length': WIKI_SEQUENCE_LENGTH,
    }
    torch.save(payload, cache)
    print('Saved:', cache, tuple(payload['discovery'].shape), tuple(payload['heldout'].shape))
    return payload

def block_batch(block, device):
    input_ids = block.unsqueeze(0) if block.ndim == 1 else block
    input_ids = input_ids.to(device, non_blocking=True)
    return {'input_ids': input_ids, 'attention_mask': torch.ones_like(input_ids)}

def statistics_forward(model, batch):
    # Hooks live inside the decoder; the LM head logits are unnecessary and expensive.
    backbone = getattr(model, 'model', model)
    return backbone(**batch, use_cache=False, return_dict=False)

def choose_statistics_batch_size(model, blocks, selected):
    candidate = min(WIKI_STATS_BATCH_SIZE, len(selected))
    if candidate < 1:
        raise ValueError('No Wikipedia blocks were selected.')
    while True:
        probe = block_batch(blocks[selected[:candidate]], input_device(model))
        try:
            with torch.inference_mode():
                output = statistics_forward(model, probe)
            del output, probe
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            return candidate
        except torch.cuda.OutOfMemoryError:
            del probe
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            if candidate == 1:
                raise
            candidate = max(1, candidate // 2)
            print(f'Wikipedia statistics OOM; retrying with batch_size={candidate}')

def collect_language_statistics(model, blocks, language, model_name, indices=None):
    layers = decoder_layers(model)
    gate_positive_counts = [None] * len(layers); gate_token_counts = [0] * len(layers)
    swiglu_abs_sums = [None] * len(layers); swiglu_token_counts = [0] * len(layers)

    def make_gate_hook(layer_index):
        def hook(_module, _inputs, output):
            gate = output.detach()
            positive = (gate > 0).sum(dim=(0, 1))
            if gate_positive_counts[layer_index] is None:
                gate_positive_counts[layer_index] = torch.zeros_like(positive, dtype=torch.int64)
            gate_positive_counts[layer_index].add_(positive)
            gate_token_counts[layer_index] += gate.shape[0] * gate.shape[1]
        return hook

    def make_swiglu_hook(layer_index):
        def hook(_module, inputs):
            x = inputs[0].detach()
            value = x.abs().sum(dim=(0, 1), dtype=torch.float32)
            if swiglu_abs_sums[layer_index] is None:
                swiglu_abs_sums[layer_index] = torch.zeros_like(value)
            swiglu_abs_sums[layer_index].add_(value)
            swiglu_token_counts[layer_index] += x.shape[0] * x.shape[1]
        return hook

    # Probe before registering hooks so the warm-up batch is not counted twice.
    selected = list(range(len(blocks))) if indices is None else list(indices)
    batch_size = choose_statistics_batch_size(model, blocks, selected)
    print(f'{model_name}:{language} statistics batch_size={batch_size}, blocks={len(selected)}')

    handles = []
    for i, layer in enumerate(layers):
        if not hasattr(layer.mlp, 'act_fn'):
            raise AttributeError('LAPE requires layer.mlp.act_fn.')
        handles.append(layer.mlp.act_fn.register_forward_hook(make_gate_hook(i)))
        handles.append(layer.mlp.down_proj.register_forward_pre_hook(make_swiglu_hook(i)))
    try:
        starts = range(0, len(selected), batch_size)
        with torch.inference_mode():
            for start in tqdm(starts, total=math.ceil(len(selected) / batch_size),
                              desc=f'{model_name}:{language}:bs{batch_size}', leave=False):
                batch_indices = selected[start:start + batch_size]
                statistics_forward(
                    model, block_batch(blocks[batch_indices], input_device(model))
                )
    finally:
        for handle in handles:
            handle.remove()
    positive_rate = torch.stack([
        value.double().cpu() / count for value, count in zip(gate_positive_counts, gate_token_counts)
    ])
    mean_abs_swiglu = torch.stack([
        (value / count).cpu() for value, count in zip(swiglu_abs_sums, swiglu_token_counts)
    ])
    return {'positive_rate': positive_rate, 'mean_abs_swiglu': mean_abs_swiglu}

def collect_model_statistics(model, model_name, indices=None):
    per_language = []
    for language in LANGUAGES:
        per_language.append(collect_language_statistics(
            model, WIKI[language]['discovery'], language, model_name, indices=indices
        ))
    return {
        'positive_rate': torch.stack([x['positive_rate'] for x in per_language], dim=-1),
        'mean_abs_swiglu': torch.stack([x['mean_abs_swiglu'] for x in per_language], dim=-1),
    }

def statistics_path(model_name):
    # Cache identity includes checkpoint, language set, and token budget.
    # This prevents silently reusing statistics from a different model or language order.
    safe = model_name.lower().replace('-', '_')
    checkpoint = (FP16_CHECKPOINT if model_name == 'FP16'
                  else Path(ACTIVE_QUANT_SPECS[model_name]['checkpoint']))
    checkpoint_tag = checkpoint.name.lower().replace('-', '_').replace('.', '_')
    language_tag = '-'.join(LANGUAGES)
    token_tag = f'{WIKI_DISCOVERY_TOKENS_PER_LANGUAGE}tokens'
    return OUTPUT_DIR / f'wikipedia_discovery_lape_stats_{safe}_{checkpoint_tag}_{language_tag}_{token_tag}.pt'



In [ ]:
# Existing FP16 13-language rates and quantized 6-language rates are reused.
def old_stats_path(model_name, language_tag):
    stem = model_name.lower().replace('-', '_')
    candidates = list(OLD_DIR.glob(
        f'wikipedia_discovery_lape_stats_{stem}_*_{language_tag}_1000000tokens.pt'
    ))
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one {model_name} stats file for {language_tag}: {candidates}')
    return candidates[0]

def checked_rates(path, expected_languages):
    payload = torch.load(path, map_location='cpu', weights_only=False)
    if (payload.get('languages') != expected_languages
            or payload.get('tokens_per_language') != WIKI_DISCOVERY_TOKENS_PER_LANGUAGE
            or payload.get('lape_activation') != 'P(SiLU(gate_proj(x)) > 0 | language)'):
        raise RuntimeError(f'Incompatible LAPE activation statistics: {path}')
    rates = payload['positive_rate'].double().cpu()
    if tuple(rates.shape) != (32, 14336, len(expected_languages)):
        raise RuntimeError(f'Unexpected LAPE statistics shape: {path}: {tuple(rates.shape)}')
    return rates

fp_old = checked_rates(old_stats_path('FP16', '-'.join(LANGUAGES)), LANGUAGES)
RATES = {'FP16': fp_old}
for model_name in ACTIVE_QUANT_METHODS:
    old = checked_rates(old_stats_path(model_name, '-'.join(OLD_LANGUAGES)), OLD_LANGUAGES)
    old_lookup = {language: old[..., i] for i, language in enumerate(OLD_LANGUAGES)}
    per_language = []
    model = None
    try:
        for language in LANGUAGES:
            if language in old_lookup:
                rates = old_lookup[language]
            else:
                cache = OUTPUT_DIR / f'positive_rate_{model_name.lower().replace("-", "_")}_{language}_all_layers.pt'
                if cache.is_file():
                    saved = torch.load(cache, map_location='cpu', weights_only=False)
                    if (saved.get('language') != language
                            or saved.get('model_name') != model_name
                            or saved.get('checkpoint') != str(QUANT_SPECS[model_name]['checkpoint'])
                            or saved.get('tokens_per_language') != WIKI_DISCOVERY_TOKENS_PER_LANGUAGE):
                        raise RuntimeError(f'Incompatible per-language LAPE cache: {cache}')
                    rates = saved['positive_rate'].double().cpu()
                else:
                    token_path = OLD_DIR / (
                        f'wikipedia_tokens_{language}_1100000_seq512.pt'
                    )
                    if not token_path.is_file():
                        raise FileNotFoundError(f'Missing original Wikipedia token blocks: {token_path}')
                    blocks = torch.load(token_path, map_location='cpu', weights_only=False)['discovery']
                    expected_blocks = WIKI_DISCOVERY_TOKENS_PER_LANGUAGE // WIKI_SEQUENCE_LENGTH
                    if tuple(blocks.shape) != (expected_blocks, WIKI_SEQUENCE_LENGTH):
                        raise RuntimeError(f'Wrong discovery blocks for {language}: {tuple(blocks.shape)}')
                    if model is None:
                        if not torch.cuda.is_available():
                            raise RuntimeError('CUDA is required to compute missing quantized LAPE statistics')
                        model = load_model(model_name)
                    stats = collect_language_statistics(model, blocks, language, model_name)
                    rates = stats['positive_rate'].double().cpu()
                    torch.save({
                        'positive_rate': rates, 'language': language,
                        'model_name': model_name,
                        'checkpoint': str(QUANT_SPECS[model_name]['checkpoint']),
                        'tokens_per_language': WIKI_DISCOVERY_TOKENS_PER_LANGUAGE,
                    }, cache)
                    print('Saved:', cache)
                    del blocks, stats
                if tuple(rates.shape) != (32, 14336):
                    raise RuntimeError(f'Wrong LAPE rate shape for {model_name}/{language}')
            per_language.append(rates)
    finally:
        if model is not None:
            del model
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
    RATES[model_name] = torch.stack(per_language, dim=-1)
    print(model_name, tuple(RATES[model_name].shape))


In [ ]:
def classify_specific_neurons_lape(positive_rate,
                                   bottom_rate=LAPE_BOTTOM_RATE,
                                   activity_quantile=ACTIVATION_PROBABILITY_QUANTILE):
    positive_rate = positive_rate.double()
    totals = positive_rate.sum(dim=-1, keepdim=True)
    distribution = torch.where(totals > 0, positive_rate / totals, 0)
    tiny = torch.finfo(torch.float64).tiny
    lape = -(distribution * distribution.clamp_min(tiny).log()).sum(dim=-1)
    normalized_lape = lape / math.log(positive_rate.size(-1))

    # Match the authors' official implementation: first exclude neurons whose
    # activation probability exceeds the global threshold for no language;
    # then take the lowest-entropy N neurons, where N is 1% of all neurons.
    probability_threshold = torch.quantile(positive_rate.flatten(), activity_quantile)
    eligible = (positive_rate > probability_threshold).any(dim=-1)
    eligible_indices = torch.where(eligible.flatten())[0]
    requested_count = max(1, round(normalized_lape.numel() * bottom_rate))
    selected_count = min(requested_count, eligible_indices.numel())
    if selected_count == 0:
        raise ValueError('No neurons passed the LAPE activation-probability filter.')
    eligible_lape = normalized_lape.flatten()[eligible_indices]
    chosen_flat = eligible_indices[eligible_lape.topk(selected_count, largest=False).indices]
    selected = torch.zeros_like(normalized_lape, dtype=torch.bool)
    selected.view(-1)[chosen_flat] = True
    membership = selected.unsqueeze(-1) & (positive_rate > probability_threshold)
    specific = membership.any(dim=-1)
    dominant = positive_rate.argmax(dim=-1)  # descriptive only; membership is multi-label.
    lape_cutoff = normalized_lape.flatten()[chosen_flat].max()
    return {
        'specific': specific,
        'membership': membership,
        'dominant': dominant,
        'lape': lape,
        'normalized_lape': normalized_lape,
        'selected_low_lape': selected,
        'eligible': eligible,
        'requested_count': requested_count,
        'selected_count': selected_count,
        'lape_cutoff': float(lape_cutoff),
        'probability_threshold': float(probability_threshold),
        'positive_rate': positive_rate,
    }



In [ ]:
# Select LAPE jointly over all 13 languages, then report 12 targets.
CLASSIFICATIONS = {
    model_name: classify_specific_neurons_lape(rates)
    for model_name, rates in RATES.items()
}
count_rows, overlap_rows = [], []
for model_name, result in CLASSIFICATIONS.items():
    for language_index, language in enumerate(LANGUAGES):
        for layer in range(32):
            count_rows.append({
                'model': model_name, 'language': language, 'layer': layer,
                'count': int(result['membership'][layer, :, language_index].sum()),
            })
pd.DataFrame(count_rows).to_csv(
    OUTPUT_DIR / 'lape_specific_neuron_counts_by_layer_language.csv', index=False
)
fp = CLASSIFICATIONS['FP16']['membership']
for method in ACTIVE_QUANT_METHODS:
    quant = CLASSIFICATIONS[method]['membership']
    for language in TARGET_LANGUAGES:
        i = LANGUAGES.index(language)
        for layer in range(32):
            fp_set = set(torch.nonzero(fp[layer, :, i], as_tuple=False).flatten().tolist())
            q_set = set(torch.nonzero(quant[layer, :, i], as_tuple=False).flatten().tolist())
            common, union = fp_set & q_set, fp_set | q_set
            overlap_rows.append({
                'method': method, 'layer': layer, 'language': language,
                'fp_count': len(fp_set), 'quant_count': len(q_set),
                'intersection': len(common),
                'jaccard': len(common) / len(union) if union else np.nan,
                'fp16_retention': len(common) / len(fp_set) if fp_set else np.nan,
            })
overlap = pd.DataFrame(overlap_rows)
if (len(overlap) != 3 * 12 * 32
        or overlap.duplicated(['method', 'language', 'layer']).any()):
    raise RuntimeError('Incomplete 12-target LAPE overlap')
overlap.to_csv(OUTPUT_DIR / 'lape_specific_neuron_layerwise_overlap.csv', index=False)
display(overlap.groupby(['method', 'language'], as_index=False)
        .agg(fp_count=('fp_count', 'sum'), quant_count=('quant_count', 'sum'),
             intersection=('intersection', 'sum')))
print('Saved 12-target LAPE counts and overlap to:', OUTPUT_DIR)
